# 03 — Model Training (Approach A)

Trains **7 ML models × 4 poverty thresholds = 28 models** on historical data (1996–2022).

**Approach A**: Predictions up to 2050 — all SSP forecast features (GDP/cap, population, HDI,
Control of Corruption, Employment in Agriculture, Gini) are available through 2050.

| Step | Description |
|------|-------------|
| Hyperparameter tuning | 5-fold CV stratified by country income quartile |
| Final fit | Full training set (1996–2015) with best params |
| Evaluation | Hold-out test set (2016–2022): RMSE, MAE, R², MAPE |
| Outputs | `models/*.pkl` (28 files), comparison CSVs + plot |


## 0. Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

from model_pipeline import (
    GPU_AVAILABLE,
    THRESHOLD_COL_MAP,
    THRESHOLD_SLUG,
    ALL_THRESHOLDS,
    MODEL_NAMES,
    PARAM_GRIDS,
    compute_metrics,
    run_full_pipeline,
    load_model_bundle,
)
from config import DATA_FINAL_DIR, MODELS_DIR, OUTPUTS_DIR

MODELS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"GPU available : {GPU_AVAILABLE}")
print(f"Thresholds    : {ALL_THRESHOLDS}")
print(f"Models        : {MODEL_NAMES}")
print(f"Total models  : {len(ALL_THRESHOLDS) * len(MODEL_NAMES)}")

## 1. Verify training artefacts

Confirm that `data/final/` contains all required files from `02_feature_engineering.ipynb`.

In [ ]:
required = ["X_train.csv", "X_test.csv", "y_train.csv", "y_test.csv",
            "cv_folds.json", "feature_names.json", "split_metadata.json"]

print("data/final/ artefacts:")
for fname in required:
    p = DATA_FINAL_DIR / fname
    status = f"{p.stat().st_size:>10,} bytes" if p.exists() else "MISSING ✗"
    print(f"  {fname:<28}  {status}")

# Quick shape check
X_train = pd.read_csv(DATA_FINAL_DIR / "X_train.csv")
X_test  = pd.read_csv(DATA_FINAL_DIR / "X_test.csv")
y_train = pd.read_csv(DATA_FINAL_DIR / "y_train.csv")
y_test  = pd.read_csv(DATA_FINAL_DIR / "y_test.csv")

print(f"\nX_train: {X_train.shape}  |  y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape}   |  y_test:  {y_test.shape}")
print(f"Features: {X_train.columns.tolist()}")

## 2. CV fold composition

Inspect the saved cross-validation folds before training.

In [ ]:
import json

with open(DATA_FINAL_DIR / "cv_folds.json") as f:
    folds_raw = json.load(f)

print(f"{'Fold':<6}  {'Train rows':>11}  {'Val rows':>9}")
for fold_id, v in folds_raw.items():
    n_tr, n_va = len(v["train_idx"]), len(v["val_idx"])
    print(f"  {fold_id}    {n_tr:>10,}  {n_va:>9,}")

# Check no overlap between train and val in any fold
for fold_id, v in folds_raw.items():
    tr, va = set(v["train_idx"]), set(v["val_idx"])
    assert len(tr & va) == 0, f"Fold {fold_id} has train/val overlap!"
print("\nAll folds: no train/val overlap ✓")

## 3. Hyperparameter grids

Show the candidates each model will search over during CV tuning.

In [ ]:
print("Hyperparameter search grids:")
total_fits = 0
for name in MODEL_NAMES:
    candidates = PARAM_GRIDS.get(name, [{}])
    n_folds    = len(folds_raw)
    n_thresholds = len(ALL_THRESHOLDS)
    fits = len(candidates) * n_folds * n_thresholds
    total_fits += fits
    print(f"  {name:<20}  {len(candidates):>2} candidates  "
          f"→ {len(candidates)*n_folds:>3} CV fits per threshold  "
          f"→ {fits:>4} total tuning fits")
    for i, p in enumerate(candidates):
        print(f"    [{i}] {p}")
    print()

print(f"Total tuning fits : {total_fits:,}")
print(f"Total final fits  : {len(MODEL_NAMES) * len(ALL_THRESHOLDS)}")

## 4. Run full training pipeline

> **Expected runtime**: 15–45 min depending on hardware (GAM and RF are the slowest).  
> Set `skip_tuning=True` for a quick smoke-test using only the first candidate per model.

In [ ]:
# Set skip_tuning=False for full hyperparameter search (recommended for final results)
# Set skip_tuning=True for a quick smoke-test (~2–5 min)
SKIP_TUNING = False

results = run_full_pipeline(
    final_dir   = DATA_FINAL_DIR,
    models_dir  = MODELS_DIR,
    outputs_dir = OUTPUTS_DIR,
    skip_tuning = SKIP_TUNING,
)

## 5. Results overview

In [ ]:
# Display the full results table
display_cols = ["model_name", "threshold", "rmse", "mae", "r2", "mape",
                "train_rmse", "elapsed_s", "gpu_used"]
print(results[display_cols].to_string(index=False))

## 6. Primary comparison — $3/day threshold

Rank models by test RMSE on the primary poverty threshold.

In [ ]:
primary = (
    results[results["threshold"] == "$3"]
    .sort_values("rmse")
    .reset_index(drop=True)
)
print("Test set performance — $3/day poverty threshold")
print("="*65)
cols = ["model_name", "rmse", "mae", "r2", "mape", "train_rmse", "elapsed_s"]
print(primary[cols].to_string(index=False))

best_model = primary.iloc[0]["model_name"]
print(f"\nBest model (lowest RMSE): {best_model}")

## 7. Overfit analysis (train vs test RMSE)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5), sharey=False)

for ax, threshold in zip(axes, ALL_THRESHOLDS):
    sub = results[results["threshold"] == threshold].sort_values("rmse")
    names    = sub["model_name"].tolist()
    x        = np.arange(len(names))
    test_r   = sub["rmse"].values
    train_r  = sub["train_rmse"].values

    ax.bar(x - 0.2, train_r, width=0.38, label="Train RMSE", color="#2ca02c", alpha=0.75)
    ax.bar(x + 0.2, test_r,  width=0.38, label="Test RMSE",  color="#d62728", alpha=0.75)
    ax.set_title(f"{threshold}/day", fontsize=9, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels([n.replace("_", "\n") for n in names], fontsize=7)
    ax.set_ylabel("RMSE (pp)", fontsize=8)
    ax.grid(True, linestyle="--", alpha=0.4, axis="y")
    if ax == axes[0]:
        ax.legend(fontsize=7)

plt.suptitle("Train vs Test RMSE — overfit check (lower bars = better)", y=1.02)
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / "overfit_check_approach_a.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Cross-threshold RMSE heatmap

Which models are most consistent across all 4 poverty thresholds?

In [ ]:
import seaborn as sns

pivot = results.pivot_table(index="model_name", columns="threshold",
                             values="rmse").reindex(
    index=MODEL_NAMES, columns=ALL_THRESHOLDS)

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(pivot, annot=True, fmt=".3f", cmap="YlOrRd",
            linewidths=0.5, annot_kws={"size": 9}, ax=ax)
ax.set_title("Test RMSE per model × threshold", fontsize=11)
ax.set_xlabel("Poverty threshold"); ax.set_ylabel("")
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / "rmse_heatmap_approach_a.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. R² comparison across thresholds

In [ ]:
pivot_r2 = results.pivot_table(index="model_name", columns="threshold",
                                values="r2").reindex(
    index=MODEL_NAMES, columns=ALL_THRESHOLDS)

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(pivot_r2, annot=True, fmt=".3f", cmap="RdYlGn",
            vmin=0, vmax=1, linewidths=0.5, annot_kws={"size": 9}, ax=ax)
ax.set_title("Test R² per model × threshold (green = better)", fontsize=11)
ax.set_xlabel("Poverty threshold"); ax.set_ylabel("")
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / "r2_heatmap_approach_a.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. MAPE by threshold — sensitivity to near-zero poverty

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for i, threshold in enumerate(ALL_THRESHOLDS):
    sub   = results[results["threshold"] == threshold].sort_values("rmse")
    names = sub["model_name"].tolist()
    mapes = sub["mape"].values
    x     = np.arange(len(names)) + i * (len(names) + 1.5)
    bars  = ax.bar(x, mapes, color=f"C{i}", alpha=0.75, label=threshold)
    ax.set_xticks(np.concatenate([np.arange(len(names)) + i*(len(names)+1.5)
                                   for i in range(len(ALL_THRESHOLDS))]))
    label_list = [n.replace("_", "\n") for n in names] * len(ALL_THRESHOLDS)

ax.set_xticks(np.concatenate([np.arange(len(MODEL_NAMES)) + i*(len(MODEL_NAMES)+1.5)
                                for i in range(len(ALL_THRESHOLDS))]))
ax.set_xticklabels(
    [n.replace("_", "\n") for n in
     results.sort_values("rmse").groupby("threshold").apply(
         lambda g: g["model_name"].tolist()).explode().tolist()[:len(MODEL_NAMES)*len(ALL_THRESHOLDS)]],
    fontsize=6.5
)
ax.set_ylabel("MAPE (%)")
ax.set_title("MAPE by model and threshold\n(MAPE only computed where poverty > 0.1%)")
ax.legend(title="Threshold")
ax.grid(True, linestyle="--", alpha=0.4, axis="y")
plt.tight_layout()
plt.show()

## 11. Verify saved model files

In [ ]:
print("Saved model files:")
model_files = sorted(MODELS_DIR.glob("*_approach_a.pkl"))
print(f"  Total: {len(model_files)} (expected: {len(MODEL_NAMES) * len(ALL_THRESHOLDS)})")
for p in model_files:
    print(f"  {p.name:<45}  {p.stat().st_size:>8,} bytes")

# Spot-check: load one model and run a prediction
print("\nSpot-check: load best model for $3 and run inference…")
best = primary.iloc[0]["model_name"]
bundle = load_model_bundle(best, "$3")
X_check = pd.read_csv(DATA_FINAL_DIR / "X_test.csv").values[:5]
preds   = bundle["model"].predict(X_check)
print(f"  Model  : {best}")
print(f"  Params : {bundle['best_params']}")
print(f"  Sample predictions (test rows 0–4): {preds.round(2)}")

## 12. Summary

| Artefact | Count | Location |
|----------|-------|----------|
| Trained models (`.pkl`) | 28 | `models/` |
| `model_comparison_approach_a.csv` | 1 | `outputs/` |
| `model_comparison_by_threshold.csv` | 1 | `outputs/` |
| `model_comparison_approach_a.png` | 1 | `outputs/` |
| `overfit_check_approach_a.png` | 1 | `outputs/` |
| `rmse_heatmap_approach_a.png` | 1 | `outputs/` |
| `r2_heatmap_approach_a.png` | 1 | `outputs/` |

**Next:** `04_explainability.ipynb` — SHAP analysis on the best-performing model  using the $3/day threshold as the primary reference.
